# CineMatch
## Eksplorasi data dan model rekomendasi film

Notebook ini membangun model **content-based filtering** menggunakan K-Nearest Neighbors (KNN) dengan cosine distance. Ekstrak CSV **MovieLens 20M** ke `ml/data/raw/`, atau gunakan `data_dir` untuk menunjuk lokasi dataset yang lain. [Dokumentasi dataset](https://files.grouplens.org/datasets/movielens/ml-20m-README.html)

Jalankan semua sel untuk memproses data, menguji peringkat rekomendasi, menyimpan artefak, dan memuat ulang model untuk contoh prediksi.

In [ ]:
from pathlib import Path
import sys

project = Path.cwd().resolve()
if (project / "CineMatch" / "ml" / "training.ipynb").is_file():
    project = project / "CineMatch"
elif project.name == "ml":
    project = project.parent
if not (project / "ml" / "training.ipynb").is_file():
    raise RuntimeError("Buka notebook dari folder CineMatch atau CineMatch/ml.")
if str(project) not in sys.path:
    sys.path.insert(0, str(project))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from ml.data_sources import DEFAULT_DATA_DIR, clean_catalog, inspect_tag_genome, scan_ratings
from ml.evaluate import evaluate
from ml.recommender import ARTIFACT_DIR, MovieRecommender
from ml.train import (
    CHUNK_SIZE, HOLDOUT_FRACTION, POSITIVE_THRESHOLD,
    build_features, make_report, save_artifacts,
)

data_dir = DEFAULT_DATA_DIR
output_dir = ARTIFACT_DIR

## 1. EDA dan pembersihan katalog

Katalog menggabungkan film, tag, serta ID IMDb dan TMDB melalui `movieId`. Proses ini membuang baris film tanpa ID atau judul, menghapus duplikasi ID, mengubah label genre yang kosong menjadi daftar kosong, dan mengambil tahun dari judul jika tersedia. Tag dinormalisasi dan dideduplikasi per pasangan film–tag. Statistik Tag Genome dilaporkan sebagai informasi dataset.

In [ ]:
catalog, catalog_stats = clean_catalog(data_dir)
genome_stats = inspect_tag_genome(data_dir, CHUNK_SIZE)
catalog_summary = pd.Series(
    {key: count for key, count in catalog_stats.items() if key != "genre_counts"},
    name="Jumlah",
)
display(catalog_summary.to_frame())
display(pd.Series(genome_stats, name="Jumlah").to_frame())
display(catalog.drop(columns="tag_text").head(5))

## 2. Distribusi rating dan pembagian kronologis

Rating dibaca per bagian, dengan batas bagian disesuaikan agar riwayat tiap pengguna tetap utuh. Setelah validasi dan deduplikasi, rating diurutkan berdasarkan waktu untuk setiap pengguna. Sebanyak 20% rating terbaru, dibulatkan ke atas, menjadi data uji. Sisa rating menjadi data preferensi awal. Untuk setiap pengguna, rating minimal 4 menandai film yang disukai.

In [ ]:
train_counts, test_cases, rating_stats = scan_ratings(
    data_dir, catalog, CHUNK_SIZE, POSITIVE_THRESHOLD, HOLDOUT_FRACTION
)
display(pd.Series(
    {key: value for key, value in rating_stats.items() if key != "rating_distribution"},
    name="Nilai",
).to_frame())

genre_distribution = pd.Series(catalog_stats["genre_counts"]).sort_values()
rating_distribution = pd.Series(rating_stats["rating_distribution"])
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), layout="constrained")
genre_distribution.plot.barh(ax=axes[0], color="#7254B3", title="Jumlah film per genre")
rating_distribution.plot.bar(ax=axes[1], color="#427D91", title="Distribusi rating", rot=0)
axes[0].set_xlabel("Film")
axes[1].set(xlabel="Rating", ylabel="Jumlah rating")
plt.show()

## 3. Ekstraksi fitur dan fitting KNN

Genre diubah menjadi vektor multi-label biner dan teks tag menjadi vektor TF-IDF. Setiap blok dinormalisasi, lalu keduanya digabung dan dinormalisasi lagi. Jika kedua film memiliki genre dan tag, kontribusinya terhadap cosine similarity adalah 75% genre dan 25% tag. Film dengan satu blok yang terisi tetap dapat dibandingkan.

KNN menyimpan matriks sparse dan mencari tetangga dengan cosine distance. Peringkat memakai skor kemiripan $1 - \text{cosine distance}$. Skor setara hingga enam desimal diurutkan menurut jumlah rating data awal, lalu ID film.

In [ ]:
features, genre_encoder, tag_vectorizer = build_features(catalog)
recommender = MovieRecommender.fit(
    features, catalog.assign(train_rating_count=train_counts)
)
print(f"Matriks fitur: {features.shape[0]:,} film × {features.shape[1]:,} fitur")
print(f"Genre: {len(genre_encoder.classes_)}")
print(f"Kosakata tag: {len(tag_vectorizer.vocabulary_):,}")
print(f"Film tanpa fitur: {np.count_nonzero(np.diff(features.indptr) == 0)}")

## 4. Evaluasi top-5 dan baseline

Evaluasi mencakup semua pengguna yang memiliki sedikitnya satu rating positif di masing-masing bagian. Film positif terakhir dari data awal menjadi input KNN, sesuai alur Discover yang meminta satu judul. Semua film yang sudah dirating di data awal dikeluarkan dari kandidat kedua metode. Baseline mengurutkan film berdasarkan jumlah rating di data awal dari seluruh pengguna. Keduanya diuji pada film positif di bagian uji.

Precision@5 mengukur proporsi rekomendasi yang relevan, Recall@5 mengukur cakupan atas film relevan di bagian uji, Hit Rate@5 menghitung pengguna dengan minimal satu hit, dan NDCG@5 memberi bobot lebih besar pada hit di posisi atas. Waktu inferensi diukur pada hingga 200 permintaan acak dengan seed tetap, tanpa memakai cache hasil pencarian. Pengukuran mencakup pemeringkatan dan pemilihan kandidat.

In [ ]:
evaluation = evaluate(recommender, test_cases)
dataset = {**catalog_stats, **rating_stats, "tag_genome": genome_stats}
report = make_report(dataset, evaluation, features, genre_encoder, tag_vectorizer)
print(f"Pengguna dievaluasi: {report['evaluated_users']:,} dari {rating_stats['users']:,}")
print(f"Judul input unik: {report['distinct_seed_movies']:,}")
print(f"Permintaan untuk pengukuran waktu per metode: {report['timing_requests_per_method']}")

labels = {
    "precision_at_5": "Precision@5",
    "recall_at_5": "Recall@5",
    "hit_rate_at_5": "Hit Rate@5",
    "ndcg_at_5": "NDCG@5",
    "mean_inference_ms": "Rata-rata (ms)",
    "p95_inference_ms": "P95 (ms)",
}
metrics = pd.DataFrame(report["metrics"]).rename(
    index=labels, columns={"knn": "KNN", "popularity": "Popularitas"}
)
display(metrics.round(4))
ax = metrics.iloc[:4].plot.bar(
    color=["#7254B3", "#427D91"], figsize=(8, 3.8), rot=0, title="Evaluasi rekomendasi top-5"
)
ax.set_ylabel("Nilai")
ax.figure.tight_layout()
plt.show()

winner = metrics.iloc[:4].idxmax(axis=1)
print("Metode dengan nilai tertinggi per metrik:")
print(winner.to_string())

### Cara membaca hasil

Evaluasi mengukur seberapa sering rekomendasi muncul kembali sebagai film yang disukai di riwayat uji. Film yang belum pernah dirating bisa tetap menarik bagi pengguna. Katalog dan tag diperlakukan sebagai satu snapshot metadata. Tag dapat berasal dari waktu setelah batas data awal pengguna, sehingga evaluasi ini merekonstruksi preferensi dari rating dengan katalog tetap. Match Score pada hasil contoh menyatakan kemiripan fitur film.

## 5. Simpan artefak dan uji model setelah dimuat ulang

File Joblib menyimpan model, encoder genre, dan vectorizer tag. Matriks sparse, katalog beserta pemetaan baris, dan hasil evaluasi disimpan bersama di `ml/artifacts`. Contoh berikut memeriksa rekomendasi untuk *Interstellar (2014)* dan memastikan filter genre berlaku setelah model dimuat dari disk.

In [ ]:
save_artifacts(output_dir, recommender, genre_encoder, tag_vectorizer, report)
loaded = MovieRecommender.load(output_dir)

seed_title = "Interstellar (2014)"
matching_ids = loaded.catalog.loc[loaded.catalog["title"].eq(seed_title), "movie_id"]
seed_id = int(matching_ids.iloc[0]) if not matching_ids.empty else int(loaded.movie_ids[0])
seed_title = loaded.catalog.iloc[loaded.movie_rows[seed_id]]["title"]
recommendations = loaded.recommend(seed_id)
genre = "Sci-Fi" if "Sci-Fi" in loaded.genre_masks else next(iter(loaded.genre_masks))
filtered = loaded.recommend(seed_id, genre=genre)

assert len(recommendations) == min(5, len(loaded.catalog) - 1)
assert len({row["movie_id"] for row in recommendations}) == len(recommendations)
assert all(row["movie_id"] != seed_id and 0 <= row["similarity_score"] <= 1
           for row in recommendations)
assert all(a["similarity_score"] >= b["similarity_score"]
           for a, b in zip(recommendations, recommendations[1:]))
assert filtered and all(genre in row["genres"] for row in filtered)

print(f"Film input: {seed_title}")
columns = ["rank", "title", "genres", "similarity_score"]
display(pd.DataFrame(recommendations)[columns])
print(f"Filter: {genre}")
display(pd.DataFrame(filtered)[columns])
print("Artefak:", ", ".join(sorted(path.name for path in output_dir.iterdir() if path.is_file())))